# 멀티카메라 비디오 처리 시스템 테스트

`videoSCST`를 사용한 멀티카메라 비디오 처리 테스트 (최적화 버전)


In [1]:
import os, sys, time, glob
import torch

# 경로 추가 (프로젝트 루트)
sys.path.append("/workspace")

from MCMT_engine.SCST.video_SCST import videoSCST

## 1. 최적화된 Args 설정


In [2]:
# === 최적화된 Args ===
class Args:
    track_thresh = 0.3
    match_thresh = 0.9
    track_buffer = 180
    mot20 = False
    cpu_workers = 16   # 최적화: 20 → 16
    chunk_sec = 30.0   # 최적화: 10 → 30 (더 큰 청크)
    batch_size = 64    # 최적화: 20 → 64 (더 큰 배치)

args = Args()

# GPU 메모리 최적화
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True
    print(f"✅ GPU 최적화 완료: {torch.cuda.get_device_name()}")
else:
    print("⚠️ CUDA를 사용할 수 없습니다. CPU 모드로 실행됩니다.")


✅ GPU 최적화 완료: NVIDIA RTX A6000


In [3]:
# 1) 도면 이미지 경로 (공통)
plan_path = "/workspace/assets/seocho/Seocho_plan_pts.png"

# 2) 카메라별 도면 기준점 (PLAN)
cam1_plan_pts = [
    (1431,1198), #측정안전통로
    (1505,1256), #노란깃발
    (1486,1068), #철근아래
    (1517,999),  #철근위
    (1505,972),  #나무적재왼쪽
    (1528,972),  #나무적재오른쪽
    (1601,1015), #범퍼
    (1664,1015), #초록깃발
    (1699,958),  #미니저수지왼쪽코너
    (1724,644),  #미니저수지위쪽코너
    (2026,449),  #출입구왼쪽
    (2032,532),  #출입구오른쪽
    (2016,615),  #컨테이너왼쪽
    (2019,654),  #컨테이너오른쪽
    (1917,939),  #미니저수지오른쪽
    (1908,625),  #미니저수지오른쪽윗코너
    (1219,1557)  #빨간꼬깔
]

cam2_plan_pts = [
    (1597,625),  #저수지 계단 아래
    (1629,619),  #저수지 계단 위
    (2023,451),  #출입구 왼쪽
    (2029,530),  #출입구 오른쪽
    (2015,615),  #컨테이너 왼쪽
    (2019,659),  #컨테이너 오른쪽
    (1372,907),  #저수지 꼭지
    (1375,952),  #꼭지 밑 안전통로
    (1505,973),  #나무적재왼쪽
    (1525,971),  #나무적재오른쪽
    (1520,998),  #철근위
    (1485,1068), #철근아래
    (1330,1177), #안전통로입구
    (1431,1198), #측정안전통로
    (1594,1176), #범퍼
    (1504,1253), #노란깃발
    (1698,960),  #작은저수지왼쪽코너
    (1915,942),  #작은저수지오른쪽코너
]

cam3_plan_pts = [
    (1597,627),  #저수지계단아래
    (1635,619),  #저수지계단위
    (1505,1256), #노란깃발
    (1695,958),  #미니저수지왼쪽아래
    (1663,1017), #초록깃발
    (1036,378),  #미니저수지오른쪽아래
    (1724,643),  #미니저수지왼쪽위
    (1910,630),  #미니저수지오른쪽 위
    (1597,1174), #범퍼
    (1616,903),  #파란깃발
    (2025,453),  #출입구왼쪽
    (2028,529),  #출입구오른쪽
    (2015,614),  #컨테이너왼쪽
    (2019,662),  #컨테이너 오른쪽
    (769,587),   #파란포장
    (1505,973),  #나무적재왼쪽
    (1531,967),  #나무적재오른쪽
    (1477,1088), #철근아래
    (1454,411),  #직원휴게실왼쪽
    (1568,377),  #직원휴게실오른쪽
]

print("✅ 캘리브레이션 포인트 설정 완료")


✅ 캘리브레이션 포인트 설정 완료


In [4]:
# 3) 카메라별 영상 좌표 (CCTV)
cam1_pts = [
    (342,661),  #측정안전통로
    (733,657),  #노란깃발
    (266,567),  #철근아래
    (230,526),  #철근위
    (129,519),  #나무적재왼쪽
    (188,514),  #나무적재오른쪽
    (811,603),  #범퍼
    (612,525),  #초록깃발
    (600,516),  #미니저수지왼쪽코너
    (177,403),  #미니저수지위쪽코너
    (420,382),  #출입구왼쪽
    (506,395),  #출입구오른쪽
    (577,421),  #컨테이너왼쪽
    (623,416),  #컨테이너오른쪽
    (979,493),  #미니저수지오른쪽
    (451,418),  #미니저수지오른쪽윗코너
    (961,717),  #빨간꼬깔
]

cam2_pts = [
    (2,462),    #저수지 계단아래
    (25,420),   #저수지 계단 위
    (107,391),  #출입구 왼쪽
    (212,399),  #출입구 오른쪽
    (309,409),  #컨테이너 왼쪽
    (360,417),  #컨테이너 오른쪽
    (313,542),  #저수지 꼭지
    (319,564),  #꼭지 밑 안전통로
    (561,503),  #나무적재왼쪽
    (580,496),  #나무적재오른쪽
    (633,506),  #철근위
    (782,530),  #철근 아래
    (966,608),  #안전통로입구
    (1055,571), #측정안전통로
    (1051,531), #범퍼
    (1206,555), #노란깃발
    (674,485),  #작은저수지왼쪽코너
    (728,459),  #작은저수지오른쪽코너
]

cam3_pts = [
    (267,350),  #저수지계단아래
    (310,305),  #저수지계단위
    (104,706),  #노란깃발
    (649,407),  #미니저수지왼쪽아래
    (664,422),  #초록깃발
    (1036,378), #미니저수지오른쪽아래
    (448,309),  #미니저수지왼쪽위
    (692,318),  #미니저수지오른쪽 위
    (539,549),  #범퍼
    (372,362),  #파란깃발
    (711,299),  #출입구왼쪽
    (763,306),  #출입구오른쪽
    (798,311),  #컨테이너왼쪽
    (827,319),  #컨테이너 오른쪽
    (769,587),  #파란포장
    (119,419),  #나무적재왼쪽
    (176,408),  #나무적재오른쪽
    (60,471),   #철근아래
    (63,301),   #직원휴게실왼쪽
    (192,291),  #직원휴게실오른쪽
]

print("✅ CCTV 좌표 설정 완료")


✅ CCTV 좌표 설정 완료


## 2. 비디오 파일 확인 및 공유 모델 초기화


In [5]:
# 테스트용 비디오 파일 경로
video_paths = [
    "/workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #1.mp4",
    "/workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #2.mp4",
    "/workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #3.mp4",
]

# 파일 존재 확인
existing_videos = []
for i, video_path in enumerate(video_paths):
    if os.path.exists(video_path):
        existing_videos.append(video_path)
        print(f"✅ Camera {i+1}: {video_path}")
    else:
        print(f"❌ Camera {i+1}: {video_path} (파일 없음)")

print(f"\n📹 총 {len(existing_videos)}개 비디오 파일 확인됨")

# 결과 저장 디렉토리 생성
os.makedirs("/workspace/results", exist_ok=True)
print("📂 결과 저장 디렉토리 준비 완료")


✅ Camera 1: /workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #1.mp4
✅ Camera 2: /workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #2.mp4
✅ Camera 3: /workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #3.mp4

📹 총 3개 비디오 파일 확인됨
📂 결과 저장 디렉토리 준비 완료


In [6]:
print("\n📦 videoSCST 초기화 중...")
scst = videoSCST(
    plan_path=plan_path,
    args=args,
    det_models=["ultra_best"],
    det_device="cuda:0",
    det_threshold=0.0,
    det_use_async=True,
    det_max_workers=1,
    # ransac_thresh=3.0  # 필요시 조정
)
print("✅ videoSCST 초기화 완료")

# 7) 각 카메라별 처리
start_time = time.time()
results = []


# 4) 카메라 구성 (CCTV 좌표, PLAN 좌표, 이름)
camera_configs = [
    (cam1_pts, cam1_plan_pts, "Camera 1"),
    (cam2_pts, cam2_plan_pts, "Camera 2"),
    (cam3_pts, cam3_plan_pts, "Camera 3"),
]
print("✅ 캘리브레이션 포인트 설정 완료")


📦 videoSCST 초기화 중...
[DEBUG] VehicleDetector loaded with Ultralytics YOLO on cuda:0
✅ videoSCST 초기화 완료
✅ 캘리브레이션 포인트 설정 완료


In [ ]:
start_time = time.time()
results = []

for i, (video_path, (cctv_pts, plan_pts, cam_name)) in enumerate(
    zip(existing_videos, camera_configs)
):
    print(f"\n📹 {cam_name} 처리 시작...")
    cam_start = time.time()

    # track_and_save()가 내부에서 calibrate(cam_pts, plan_pts) → H 계산/캐시 후,
    # 1) 카메라 뷰 추적 결과 저장
    # 2) 도면(미니맵) 결과 저장 (PlanRenderer 사용)
    result = scst.track_and_save(
        video_path=video_path,
        cam_pts=cctv_pts,
        plan_pts=plan_pts,
        plan_img_path=plan_path,
        camera_save_path=f"/workspace/results/tracking_result{i+1}.mp4",
        plan_save_path=f"/workspace/results/plan_result{i+1}.mp4",
        # ransac_thresh=3.0,    # 필요시 per-call로 조정 가능
        # cam_trail_len=30      # 카메라뷰 시각화 궤적 길이(내부 비주얼라이저)
    )

    results.append(result)
    cam_time = time.time() - cam_start
    print(f"✅ {cam_name} 처리 완료 ({cam_time:.2f}초, {len(result)} 프레임)")


📹 Camera 1 처리 시작...
[INFO] Camera calibration completed. H:
[[     2.5807     -10.255     -650.72]
 [      2.236     -16.418      4560.6]
 [  0.0023825  -0.0097397           1]]


## 4. 결과 요약 및 리소스 정리


In [ ]:
# 8) 결과 요약
total_time = time.time() - start_time
total_frames = sum(len(r) for r in results)

print(f"\n🎉 모든 카메라 처리 완료!")
print(f"⏱️  총 처리 시간: {total_time:.2f}초")
print(f"📊 총 처리 프레임: {total_frames:,}개")
print(f"🚀 평균 처리 속도: {total_frames/total_time:.1f} FPS")
print(f"💾 결과 저장 위치: /workspace/results/")

# 9) 결과 파일 확인
result_files = glob.glob("/workspace/results/*.mp4")
print(f"\n📁 생성된 결과 파일 ({len(result_files)}개):")
for fp in sorted(result_files):
    size_mb = os.path.getsize(fp) / (1024*1024)
    print(f"  - {os.path.basename(fp)}  ({size_mb:.1f} MB)")

print("\n✅ 완료")